In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/global_ads_performance_dataset.csv")

df.head()

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1800 non-null   str    
 1   platform       1800 non-null   str    
 2   campaign_type  1800 non-null   str    
 3   industry       1800 non-null   str    
 4   country        1800 non-null   str    
 5   impressions    1800 non-null   int64  
 6   clicks         1800 non-null   int64  
 7   CTR            1800 non-null   float64
 8   CPC            1800 non-null   float64
 9   ad_spend       1800 non-null   float64
 10  conversions    1800 non-null   int64  
 11  CPA            1800 non-null   float64
 12  revenue        1800 non-null   float64
 13  ROAS           1800 non-null   float64
dtypes: float64(6), int64(3), str(5)
memory usage: 263.9 KB


Dữ liệu gồm 1.800 bản ghi, 14 cột; các cột đều có 1.800 giá trị không null

In [ ]:
# Tạo bản Working riêng, chỉ chuyển kiểu date, không xóa bản ghi
df_working = df.copy()

df_working["date"] = pd.to_datetime(
    df_working["date"],
    errors="coerce"
)

In [ ]:
# Kiểm tra tính duy nhất của tổ hợp 5 chiều 
candidate_key = [
    "date",
    "platform",
    "campaign_type",
    "industry",
    "country"
]

key_check = pd.DataFrame({
    "Total Records": [len(df_working)],
    "Unique Keys": [
        df_working[candidate_key].drop_duplicates().shape[0]
    ],
    "Duplicate Keys": [
        df_working.duplicated(subset=candidate_key).sum()
    ],
    "Exact Duplicates": [
        df_working.duplicated().sum()
    ]
})

display(key_check)

,Total Records,Unique Keys,Duplicate Keys,Exact Duplicates
0,1800,1788,12,0


Có 1.788 tổ hợp khóa đề xuất duy nhất, 12 tổ hợp lặp liên quan 24 bản ghi; 0 dòng trùng hoàn toàn. Chưa thể khẳng định Grain khi thiếu Campaign ID.

In [5]:
duplicate_groups = df_working[
    df_working.duplicated(subset=candidate_key, keep=False)
].sort_values(by=candidate_key)

display(duplicate_groups)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
192,2024-01-04,Meta Ads,Shopping,EdTech,UK,117217,2133,0.0182,1.29,2751.57,129,21.33,25455.36,9.25
760,2024-01-04,Meta Ads,Shopping,EdTech,UK,184729,2586,0.0140,0.90,2327.40,201,11.58,16864.93,7.25
170,2024-02-20,Google Ads,Shopping,Fintech,UK,162489,4500,0.0277,3.03,13635.00,184,74.10,54944.54,4.03
1742,2024-02-20,Google Ads,Shopping,Fintech,UK,89431,3300,0.0369,1.47,4851.00,56,86.62,9891.88,2.04
365,2024-02-27,TikTok Ads,Search,SaaS,UAE,7637,415,0.0544,1.54,639.10,10,63.91,1137.83,1.78
578,2024-02-27,TikTok Ads,Search,SaaS,UAE,195972,13796,0.0704,0.50,6898.00,645,10.69,54701.33,7.93
842,2024-04-04,TikTok Ads,Search,SaaS,UK,45382,2246,0.0495,1.59,3571.14,46,77.63,2662.90,0.75
863,2024-04-04,TikTok Ads,Search,SaaS,UK,199291,16660,0.0836,1.16,19325.60,1016,19.02,207611.26,10.74
14,2024-04-22,TikTok Ads,Shopping,EdTech,Canada,37711,2111,0.0560,0.68,1435.48,92,15.60,13898.86,9.68
1495,2024-04-22,TikTok Ads,Shopping,EdTech,Canada,186006,9002,0.0484,1.52,13683.04,266,51.44,31194.26,2.28


In [23]:
# Mô phỏng loại dòng trùng lặp
grain_cols = [
    "date", "platform", "campaign_type",
    "industry", "country"
]

df_wrong = df_working.drop_duplicates(
    subset=grain_cols,
    keep="first"
)

metrics = [
    "impressions", "clicks", "conversions",
    "ad_spend", "revenue"
]

impact = pd.DataFrame({
    "Metric": ["Row Count"] + metrics,
    "Original": [len(df_working)] +
                [df_working[col].sum() for col in metrics],
    "Wrong Deduplication": [len(df_wrong)] +
                           [df_wrong[col].sum() for col in metrics]
})

impact["Lost Value"] = (
    impact["Original"] - impact["Wrong Deduplication"]
)

impact["Loss (%)"] = (
    impact["Lost Value"] / impact["Original"] * 100
).round(4)

display(impact)

,Metric,Original,Wrong Deduplication,Lost Value,Loss (%)
0,Row Count,1.800000e+03,1.788000e+03,12.00,0.6667
1,impressions,1.852542e+08,1.835391e+08,1715158.00,0.9258
2,clicks,7.132816e+06,7.050517e+06,82299.00,1.1538
3,conversions,3.268120e+05,3.228100e+05,4002.00,1.2246
4,ad_spend,1.110875e+07,1.100037e+07,108377.92,0.9756
5,revenue,5.418333e+07,5.361959e+07,563745.69,1.0404


Nếu loại trùng theo tổ hợp khóa đề xuất, mất 12/1.800 dòng (0,6667%); giảm 1.715.158 impressions (0,9258%), 82.299 clicks (1,1538%), 4.002 conversions (1,2246%), 108.377,92 ad_spend (0,9756%) và 563.745,69 revenue (1,0404%)

In [8]:
date_coverage = pd.DataFrame({
    "Min Date": [df_working["date"].min()],
    "Max Date": [df_working["date"].max()],
    "Unique Dates": [df_working["date"].nunique()],
    "Unique Months": [
        df_working["date"].dt.to_period("M").nunique()
    ]
})

display(date_coverage)

,Min Date,Max Date,Unique Dates,Unique Months
0,2024-01-01,2024-12-30,360,12


In [24]:
categorical_cols = [
    "platform",
    "campaign_type",
    "industry",
    "country"
]

for col in categorical_cols:
    display(
        df_working[col].value_counts().rename_axis(col).reset_index(name="Records")
    )

,platform,Records
0,Google Ads,720
1,Meta Ads,630
2,TikTok Ads,450


,campaign_type,Records
0,Search,477
1,Video,456
2,Shopping,447
3,Display,420


,industry,Records
0,EdTech,372
1,SaaS,370
2,Fintech,361
3,E-commerce,349
4,Healthcare,348


,country,Records
0,UK,266
1,USA,266
2,Canada,262
3,India,261
4,UAE,258
5,Germany,255
6,Australia,232


In [10]:
# 4.1.4 - Temporal Coverage

df_coverage = df_working.copy()
df_coverage["date"] = pd.to_datetime(df_coverage["date"])
df_coverage["month"] = df_coverage["date"].dt.to_period("M")

all_dates = pd.date_range(
    df_coverage["date"].min(),
    df_coverage["date"].max(),
    freq="D"
)

all_months = pd.period_range(
    df_coverage["month"].min(),
    df_coverage["month"].max(),
    freq="M"
)

print("Date range:",
      df_coverage["date"].min().date(),
      "to",
      df_coverage["date"].max().date())
print("Days with data:", df_coverage["date"].nunique())
print("Months with data:", df_coverage["month"].nunique())

temporal_results = []

for level, col, periods in [
    ("Date", "date", all_dates),
    ("Month", "month", all_months)
]:
    coverage = (
        df_coverage.groupby(["platform", col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=periods, fill_value=0)
    )

    for platform, row in coverage.iterrows():
        days_with_data = (row > 0).sum()

        temporal_results.append({
            "Level": level,
            "Platform": platform,
            "Periods with data": days_with_data,
            "Total periods": len(periods),
            "Coverage (%)": round(
                days_with_data / len(periods) * 100, 2
            )
        })

display(pd.DataFrame(temporal_results))

Date range: 2024-01-01 to 2024-12-30
Days with data: 360
Months with data: 12


,Level,Platform,Periods with data,Total periods,Coverage (%)
0,Date,Google Ads,307,365,84.11
1,Date,Meta Ads,308,365,84.38
2,Date,TikTok Ads,258,365,70.68
3,Month,Google Ads,12,12,100.00
4,Month,Meta Ads,12,12,100.00
5,Month,TikTok Ads,12,12,100.00


In [11]:
# 4.1.4 - Dimensional Coverage

dimension_groups = {
    "Platform × Campaign Type": ["platform", "campaign_type"],
    "Platform × Industry": ["platform", "industry"],
    "Platform × Country": ["platform", "country"]
}

dimension_results = []

for name, cols in dimension_groups.items():
    observed = df_coverage.groupby(cols).size()

    possible = 1
    for col in cols:
        possible *= df_coverage[col].nunique()

    dimension_results.append({
        "Combination": name,
        "Observed": len(observed),
        "Possible": possible,
        "Coverage (%)": round(
            len(observed) / possible * 100, 2
        ),
        "Min records/group": observed.min(),
        "Median records/group": observed.median(),
        "Max records/group": observed.max()
    })

display(pd.DataFrame(dimension_results))

,Combination,Observed,Possible,Coverage (%),Min records/group,Median records/group,Max records/group
0,Platform × Campaign Type,12,12,100.0,101,160.0,192
1,Platform × Industry,15,15,100.0,77,127.0,162
2,Platform × Country,21,21,100.0,55,89.0,121


In [12]:
# 4.1.4 - Detailed Coverage & Data Sparsity

detail_cols = [
    "platform",
    "campaign_type",
    "industry",
    "country"
]

detail_counts = (
    df_coverage.groupby(detail_cols)
    .size()
    .reset_index(name="record_count")
)

possible = 1
for col in detail_cols:
    possible *= df_coverage[col].nunique()

counts = detail_counts["record_count"]

detail_summary = pd.DataFrame({
    "Metric": [
        "Possible combinations",
        "Observed combinations",
        "Missing combinations",
        "Coverage (%)",
        "Min records/group",
        "Median records/group",
        "Max records/group",
        "Combinations with 1 record",
        "Combinations with <5 records",
        "Combinations with >=5 records"
    ],
    "Value": [
        possible,
        len(counts),
        possible - len(counts),
        round(len(counts) / possible * 100, 2),
        counts.min(),
        counts.median(),
        counts.max(),
        (counts == 1).sum(),
        (counts < 5).sum(),
        (counts >= 5).sum()
    ]
})

display(detail_summary)

,Metric,Value
0,Possible combinations,420.0
1,Observed combinations,407.0
2,Missing combinations,13.0
3,Coverage (%),96.9
4,Min records/group,1.0
5,Median records/group,4.0
6,Max records/group,13.0
7,Combinations with 1 record,28.0
8,Combinations with <5 records,234.0
9,Combinations with >=5 records,173.0


Từ 01/01–30/12/2024, có dữ liệu 360/365 ngày, đủ 12 tháng. Theo ngày: Google 307/365 (84,11%), Meta 308/365 (84,38%), TikTok 258/365 (70,68%); theo tháng cả ba đạt 12/12. Tổ hợp Platform × Campaign Type 12/12, × Industry 15/15, × Country 21/21. Tổ hợp 4 chiều có 407/420 (96,90%), thiếu 13; 234/407 nhóm dưới 5 bản ghi, trong đó 28 nhóm chỉ 1 bản ghi.

In [13]:
# 4.2.1 - Missing Values & Duplicate Records

dq_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Missing values",
        "Exact duplicate records",
        "Records with missing values"
    ],
    "Value": [
        len(df_working),
        len(df_working.columns),
        int(df_working.isna().sum().sum()),
        int(df_working.duplicated().sum()),
        int(df_working.isna().any(axis=1).sum())
    ]
})

display(dq_summary)

missing_by_column = pd.DataFrame({
    "Column": df_working.columns,
    "Missing count": df_working.isna().sum().values,
    "Missing (%)": (
        df_working.isna().mean().values * 100
    ).round(2)
})

display(missing_by_column)

,Metric,Value
0,Total records,1800
1,Total columns,14
2,Missing values,0
3,Exact duplicate records,0
4,Records with missing values,0


,Column,Missing count,Missing (%)
0,date,0,0.0
1,platform,0,0.0
2,campaign_type,0,0.0
3,industry,0,0.0
4,country,0,0.0
5,impressions,0,0.0
6,clicks,0,0.0
7,CTR,0,0.0
8,CPC,0,0.0
9,ad_spend,0,0.0


Không có giá trị thiếu, không có bản ghi trùng hoàn toàn

In [14]:
# 4.2.1 - Data Validity

validity_checks = {
    "Invalid dates": df_working["date"].isna().sum(),
    "Negative impressions": (df_working["impressions"] < 0).sum(),
    "Negative clicks": (df_working["clicks"] < 0).sum(),
    "Negative conversions": (df_working["conversions"] < 0).sum(),
    "Negative ad_spend": (df_working["ad_spend"] < 0).sum(),
    "Negative revenue": (df_working["revenue"] < 0).sum(),
    "Negative CPC": (df_working["CPC"] < 0).sum(),
    "Negative CPA": (df_working["CPA"] < 0).sum(),
    "Negative ROAS": (df_working["ROAS"] < 0).sum(),
    "Clicks > impressions": (
        df_working["clicks"] > df_working["impressions"]
    ).sum(),
    "CTR outside [0, 1]": (
        (df_working["CTR"] < 0) |
        (df_working["CTR"] > 1)
    ).sum(),
    "Conversions > 0 when clicks = 0": (
        (df_working["conversions"] > 0) &
        (df_working["clicks"] == 0)
    ).sum(),
    "CPC > 0 when clicks = 0": (
        (df_working["CPC"] > 0) &
        (df_working["clicks"] == 0)
    ).sum(),
    "CPA > 0 when conversions = 0": (
        (df_working["CPA"] > 0) &
        (df_working["conversions"] == 0)
    ).sum(),
    "ROAS > 0 when ad_spend = 0": (
        (df_working["ROAS"] > 0) &
        (df_working["ad_spend"] == 0)
    ).sum()
}

validity_results = pd.DataFrame({
    "Validation rule": validity_checks.keys(),
    "Violations": validity_checks.values()
})

display(validity_results)

,Validation rule,Violations
0,Invalid dates,0
1,Negative impressions,0
2,Negative clicks,0
3,Negative conversions,0
4,Negative ad_spend,0
5,Negative revenue,0
6,Negative CPC,0
7,Negative CPA,0
8,Negative ROAS,0
9,Clicks > impressions,0


In [25]:
from decimal import Decimal

kpi_columns = ["CTR", "CPC", "CPA", "ROAS"]
raw_strings = pd.read_csv("../data/raw/global_ads_performance_dataset.csv", dtype=str)

def decimal_places(value):
    if pd.isna(value):
        return None
    return max(0, -Decimal(str(value)).as_tuple().exponent)

precision_summary = pd.DataFrame([
    {
        "KPI": col,
        "Decimal Places Distribution": dict(raw_strings[col].map(decimal_places).value_counts().sort_index()),
        "Max Decimal Places": raw_strings[col].map(decimal_places).max()
    }
    for col in kpi_columns
])
display(precision_summary)


,KPI,Decimal Places Distribution,Max Decimal Places
0,CTR,"{2: 21, 3: 177, 4: 1602}",4
1,CPC,"{1: 186, 2: 1614}",2
2,CPA,"{1: 189, 2: 1611}",2
3,ROAS,"{1: 181, 2: 1619}",2


CTR tối đa 4 chữ số thập phân (1.602 dòng có 4 chữ số); CPC, CPA, ROAS tối đa 2 chữ số (lần lượt 1.614, 1.611, 1.619 dòng có 2 chữ số). Ngưỡng kiểm tra được đặt 0,00005 cho CTR và 0,005 cho các KPI còn lại theo giả định làm tròn thông thường

In [29]:
# 4.2.1 - KPI Consistency

import numpy as np

kpi_formulas = {
    "CTR": ("clicks", "impressions"),
    "CPC": ("ad_spend", "clicks"),
    "CPA": ("ad_spend", "conversions"),
    "ROAS": ("revenue", "ad_spend")
}

tolerances = {
    "CTR": 0.00005,
    "CPC": 0.005,
    "CPA": 0.005,
    "ROAS": 0.005
}

kpi_results = []

for kpi, (numerator, denominator) in kpi_formulas.items():
    calculated = np.divide(
        df_working[numerator],
        df_working[denominator],
        out=np.zeros(len(df_working), dtype=float),
        where=df_working[denominator].to_numpy() != 0
    )

    difference = np.abs(df_working[kpi] - calculated)

    kpi_results.append({
        "KPI": kpi,
        "Tolerance": tolerances[kpi],
        "Inconsistent records": int(
            (difference > tolerances[kpi] + 1e-10).sum()
        ),
        "Max difference": round(float(difference.max()), 8)
    })

display(pd.DataFrame(kpi_results))

,KPI,Tolerance,Inconsistent records,Max difference
0,CTR,0.00005,47,0.000177
1,CPC,0.00500,0,0.000000
2,CPA,0.00500,0,0.005000
3,ROAS,0.00500,0,0.004995


CTR 47/1.800 bản ghi (2,61%) vượt ngưỡng 0,00005, sai lệch lớn nhất khoảng 0,000177. CPC, CPA, ROAS đều 0 bản ghi vượt ngưỡng 0,005. Nguyên nhân sai lệch CTR chưa được xác minh.

In [31]:
ctr_check = df_working[["date", "platform", "impressions", "clicks", "CTR"]].copy()
ctr_check["CTR_Calculated"] = (
    ctr_check["clicks"] / ctr_check["impressions"].replace(0, np.nan)
)
ctr_check["Absolute_Difference"] = (
    ctr_check["CTR"] - ctr_check["CTR_Calculated"]
).abs()
ctr_check["Mismatch"] = ctr_check["Absolute_Difference"] > tolerances["CTR"] + 1e-10
ctr_mismatch = ctr_check[ctr_check["Mismatch"]]
display(ctr_mismatch.sort_values("Absolute_Difference", ascending=False).head(10))


,date,platform,impressions,clicks,CTR,CTR_Calculated,Absolute_Difference,Mismatch
1767,2024-03-21,TikTok Ads,5497,231,0.0422,0.042023,0.000177,True
1721,2024-01-22,TikTok Ads,5059,415,0.0822,0.082032,0.000168,True
36,2024-01-05,TikTok Ads,6015,410,0.0683,0.068163,0.000137,True
1765,2024-11-30,TikTok Ads,7234,496,0.0687,0.068565,0.000135,True
522,2024-04-25,Meta Ads,6074,128,0.0212,0.021073,0.000127,True
1481,2024-09-01,TikTok Ads,6628,261,0.0395,0.039378,0.000122,True
859,2024-11-02,Google Ads,7802,340,0.0437,0.043579,0.000121,True
464,2024-12-08,Meta Ads,6875,158,0.0231,0.022982,0.000118,True
278,2024-07-05,Google Ads,6881,357,0.0520,0.051882,0.000118,True
1051,2024-05-01,Meta Ads,7196,287,0.0400,0.039883,0.000117,True


In [32]:
ctr_check["Impression_Group"] = pd.cut(
    ctr_check["impressions"],
    bins=[0, 10000, 50000, 100000, float("inf")],
    labels=["<10K", "10K-50K", "50K-100K", ">100K"],
    include_lowest=True
)
ctr_analysis = ctr_check.groupby("Impression_Group", observed=True).agg(
    Total_Records=("Mismatch", "size"),
    Mismatch_Count=("Mismatch", "sum"),
    Max_Difference=("Absolute_Difference", "max")
)
ctr_analysis["Mismatch_Rate (%)"] = (
    ctr_analysis["Mismatch_Count"] / ctr_analysis["Total_Records"] * 100
).round(2)
display(ctr_analysis)


,Total_Records,Mismatch_Count,Max_Difference,Mismatch_Rate (%)
Impression_Group,,,,
<10K,41,29,0.000177,70.73
10K-50K,369,18,0.000095,4.88
50K-100K,461,0,0.000019,0.00
>100K,929,0,0.000009,0.00


Trong 47 sai lệch CTR, nhóm dưới 10.000 impressions có 29/41 dòng sai lệch (70,73%); nhóm 10.000–50.000 có 18/369 (4,88%); các nhóm từ 50.000 trở lên không có sai lệch vượt ngưỡng. Giữ CTR gốc để đối chiếu, dùng CTR tính lại cho phân tích và chờ xác minh tại nguồn.

In [33]:
# KPI Aggregation Consistency

kpi_formulas = {
    "CTR": ("clicks", "impressions"),
    "CPC": ("ad_spend", "clicks"),
    "CPA": ("ad_spend", "conversions"),
    "ROAS": ("revenue", "ad_spend")
}

aggregation_results = []

for kpi, (num, den) in kpi_formulas.items():
    ratio_of_sums = (
        df_working[num].sum() / df_working[den].sum()
        if df_working[den].sum() != 0 else float("nan")
    )

    aggregation_results.append({
        "KPI": kpi,
        "Ratio of Sums": round(ratio_of_sums, 6),
        "Average of Row KPI": round(df_working[kpi].mean(), 6)
    })

display(pd.DataFrame(aggregation_results))

,KPI,Ratio of Sums,Average of Row KPI
0,CTR,0.038503,0.038427
1,CPC,1.557414,1.572756
2,CPA,33.991252,46.608961
3,ROAS,4.877537,6.450367


In [35]:
numeric_cols = ["impressions", "clicks", "ad_spend", "conversions", "revenue", "CTR", "CPC", "CPA", "ROAS"]

Q1 = df_working[numeric_cols].quantile(0.25)
Q3 = df_working[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (df_working[numeric_cols] < lower_bound) | (df_working[numeric_cols] > upper_bound)

outlier_summary = pd.DataFrame({
    "Outlier Count": outlier_mask.sum(),
    "Outlier Rate (%)": (outlier_mask.mean() * 100).round(2),
    "Min": df_working[numeric_cols].min(),
    "Max": df_working[numeric_cols].max()
})

display(outlier_summary)

,Outlier Count,Outlier Rate (%),Min,Max
impressions,0,0.00,5059.0000,199650.0000
clicks,42,2.33,91.0000,16660.0000
ad_spend,90,5.00,58.0000,38453.3200
conversions,85,4.72,2.0000,1151.0000
revenue,127,7.06,142.6900,295028.2600
CTR,12,0.67,0.0089,0.0956
CPC,7,0.39,0.2800,3.9500
CPA,140,7.78,4.8000,335.8600
ROAS,123,6.83,0.1300,49.0000


In [36]:
outlier_detail = pd.DataFrame({
    "Lower Bound": lower_bound,
    "Upper Bound": upper_bound,
    "Below Lower": (df_working[numeric_cols] < lower_bound).sum(),
    "Above Upper": (df_working[numeric_cols] > upper_bound).sum()
})

display(outlier_detail.round(4))

,Lower Bound,Upper Bound,Below Lower,Above Upper
impressions,-88335.3750,293753.6250,0,0
clicks,-4247.0000,11553.0000,0,42
ad_spend,-7767.2763,18189.6938,0,90
conversions,-230.8750,542.1250,0,85
revenue,-40255.6838,86494.8263,0,127
CTR,-0.0112,0.0864,0,12
CPC,-0.7000,3.7000,0,7
CPA,-34.7125,111.7275,0,140
ROAS,-6.8938,17.2762,0,123


In [37]:
# Raw → Working Reconciliation

metrics = [
    "impressions", "clicks", "conversions",
    "ad_spend", "revenue"
]

reconciliation = pd.DataFrame({
    "Metric": ["Row Count"] + metrics,
    "Raw": [len(df)] + [df[col].sum() for col in metrics],
    "Working": [len(df_working)] +
               [df_working[col].sum() for col in metrics]
})

reconciliation["Difference"] = (
    reconciliation["Working"] - reconciliation["Raw"]
).round(6)

display(reconciliation)

,Metric,Raw,Working,Difference
0,Row Count,1.800000e+03,1.800000e+03,0.0
1,impressions,1.852542e+08,1.852542e+08,0.0
2,clicks,7.132816e+06,7.132816e+06,0.0
3,conversions,3.268120e+05,3.268120e+05,0.0
4,ad_spend,1.110875e+07,1.110875e+07,0.0
5,revenue,5.418333e+07,5.418333e+07,0.0


Raw và Working cùng 1.800 dòng; tổng impressions, clicks, conversions, ad_spend, revenue đều có chênh lệch bằng 0.